# ResidualBlock Test Suite

From-scratch Mamba `ResidualBlock` implementation tests.
Designed to run in **Google Colab with a GPU runtime**
(Runtime → Change runtime type → T4 GPU).

| Section | What it tests |
|---------|--------------|
| 1. Architecture Inspection | shapes, parameter breakdown |
| 2. Correctness Tests | stability, causality, gradient flow |
| 3. SSM Internals | A-matrix decay, Δ distribution, hidden state dynamics |
| 4. AR(1) Learning Task | can the SSM learn temporal memory? |


In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────
%pip install einops accelerated-scan -q


In [ ]:
import os, sys, math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from einops import rearrange

# ── accelerated_scan import with fallback ─────────────────────────────────
# warp   = fastest (CUDA JIT kernel, requires nvcc — may fail in Colab)
# scalar = Triton kernel (ships with PyTorch 2.x, no nvcc needed)
# ref    = pure PyTorch (always works, slowest)
try:
    from accelerated_scan.warp import scan as warp_scan
    _scan_backend = "warp (CUDA JIT)"
except (ImportError, OSError):
    try:
        from accelerated_scan.scalar import scan as warp_scan
        _scan_backend = "scalar (Triton)"
    except (ImportError, OSError):
        from accelerated_scan.ref import scan as warp_scan
        _scan_backend = "ref (pure PyTorch)"

# ── Path setup ────────────────────────────────────────────────────────────
# Adjust if the notebook is not in the repo root.
sys.path.insert(0, os.path.join(os.getcwd(), "mamba"))
from mamba_block import MambaBlock, ResidualBlock, RMSNorm

# ── Device ────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device:       {device}")
print(f"Scan backend: {_scan_backend}")


In [ ]:
@dataclass
class MambaConfig:
    d_input:     int  = 64
    d_model:     int  = 128   # inner expansion (2x d_input)
    d_state:     int  = 16    # SSM state dimension N
    dt_rank:     int  = 8     # rank of delta projection (~d_model/16)
    kernel_size: int  = 4     # causal conv kernel
    bias:        bool = False
    conv_bias:   bool = True

config = MambaConfig()
block  = ResidualBlock(config).to(device)
block.eval()
print(block)
print(f"\nTotal parameters: {sum(p.numel() for p in block.parameters()):,}")


## 1. Architecture Inspection

In [ ]:
# ── Test 1: output shape must match input shape ───────────────────────────
test_shapes = [
    (1,  32, config.d_input),   # single item, short
    (4,  64, config.d_input),   # small batch, medium sequence
    (8, 128, config.d_input),   # larger batch, long sequence
]

header = f"{'Input':>24}  {'Output':>24}  Result"
print(header)
print("-" * len(header))

all_pass = True
for shape in test_shapes:
    x = torch.randn(*shape, device=device)
    with torch.no_grad():
        y = block(x)
    ok = x.shape == y.shape
    all_pass = all_pass and ok
    print(f"{str(tuple(x.shape)):>24}  {str(tuple(y.shape)):>24}  {'PASS' if ok else 'FAIL'}")

print(f"\nAll shape tests: {'PASSED' if all_pass else 'FAILED'}")


In [ ]:
# ── Test 2: parameter breakdown by component ─────────────────────────────
mb = block.mamba_block

components = {
    "input_proj":    mb.input_proj,
    "res_proj":      mb.res_proj,
    "conv1d":        mb.conv1d,
    "x_B_proj":      mb.x_B_proj,
    "x_C_proj":      mb.x_C_proj,
    "x_dt_proj":     mb.x_dt_proj,
    "dt_proj":       mb.dt_proj,
    "output_proj":   mb.output_proj,
    "norm (RMSNorm)": block.norm,
}
counts = {k: sum(p.numel() for p in v.parameters()) for k, v in components.items()}
counts["A_log"] = mb.A_log.numel()
counts["D"]     = mb.D.numel()
total = sum(p.numel() for p in block.parameters())

print(f"{'Component':<22} {'Params':>9}  {'Share':>6}")
print("-" * 42)
for k, n in counts.items():
    print(f"{k:<22} {n:>9,}  {100 * n / total:>5.1f}%")
print("-" * 42)
print(f"{'TOTAL':<22} {total:>9,}  100.0%")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
names = list(counts.keys())
vals  = list(counts.values())
colors = plt.cm.tab10(np.linspace(0, 1, len(names)))
bars = ax.barh(names, vals, color=colors)
for bar, v in zip(bars, vals):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
            f"{v:,}", va="center", fontsize=8)
ax.set_xlabel("Parameters")
ax.set_title("ResidualBlock — Parameter Distribution")
ax.set_xlim(0, max(vals) * 1.2)
plt.tight_layout()
plt.show()


## 2. Correctness Tests

In [ ]:
# ── Test 3: numerical stability ───────────────────────────────────────────
cases = {
    "normal":           torch.randn(2, 64, config.d_input),
    "zeros":            torch.zeros(2, 64, config.d_input),
    "large  (x100)":    torch.randn(2, 64, config.d_input) * 100,
    "small  (x1e-4)":   torch.randn(2, 64, config.d_input) * 1e-4,
    "long seq (L=256)": torch.randn(2, 256, config.d_input),
}

print(f"{'Case':<22} {'NaN':>5} {'Inf':>5} {'Max|y|':>12}  Result")
print("-" * 55)
all_ok = True
for name, x in cases.items():
    with torch.no_grad():
        y = block(x.to(device))
    nan = torch.isnan(y).any().item()
    inf = torch.isinf(y).any().item()
    ok  = not nan and not inf
    all_ok = all_ok and ok
    print(f"{name:<22} {str(nan):>5} {str(inf):>5} "
          f"{y.abs().max().item():>12.4f}  {'PASS' if ok else 'FAIL'}")

print(f"\nNumerical stability: {'ALL PASSED' if all_ok else 'SOME FAILED'}")


In [ ]:
# ── Test 4: causality ─────────────────────────────────────────────────────
# Perturbing positions [t_p, L) must NOT change outputs at [0, t_p).

torch.manual_seed(0)
L, t_p = 64, 32

x_orig = torch.randn(1, L, config.d_input, device=device)
x_pert = x_orig.clone()
x_pert[:, t_p:] += torch.randn(1, L - t_p, config.d_input, device=device)

with torch.no_grad():
    y_orig = block(x_orig)
    y_pert = block(x_pert)

diff     = (y_orig - y_pert).abs().mean(-1).squeeze().cpu().numpy()  # (L,)
pre_max  = float(diff[:t_p].max())
post_max = float(diff[t_p:].max())
causal   = pre_max < 1e-5

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(diff, color="steelblue", linewidth=1.5)
ax.axvline(t_p, color="red", linestyle="--", linewidth=1.5,
           label=f"Perturbation at t={t_p}")
ymax = max(diff.max() * 1.15, 1e-7)
ax.fill_betweenx([0, ymax], 0,   t_p, alpha=0.08, color="green",
                 label="Past — must be 0")
ax.fill_betweenx([0, ymax], t_p, L,   alpha=0.08, color="red",
                 label="Future — can change")
ax.set_ylim(0, ymax)
ax.set_xlabel("Sequence position")
ax.set_ylabel("Mean |Δ output|")
ax.set_title(f"Causality Test — {'PASSED' if causal else 'FAILED'}"
             f"  (max pre-perturbation diff: {pre_max:.2e})")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Max diff BEFORE t={t_p}: {pre_max:.2e}  ({'PASS' if causal else 'FAIL'})")
print(f"Max diff AFTER  t={t_p}: {post_max:.4f}")


In [ ]:
# ── Test 5: gradient flow ─────────────────────────────────────────────────
# Every parameter must receive a non-zero gradient on a typical forward pass.

block.train()
x = torch.randn(2, 64, config.d_input, device=device)
block(x).mean().backward()

print(f"{'Parameter':<32} {'Shape':<22} {'Has Grad':>9} {'Grad Norm':>12}")
print("-" * 80)
no_grad = []
for name, p in block.named_parameters():
    hg   = p.grad is not None
    norm = p.grad.norm().item() if hg else 0.0
    if not hg:
        no_grad.append(name)
    print(f"{name:<32} {str(tuple(p.shape)):<22} "
          f"{'YES' if hg else 'NO ':>9} {norm:>12.6f}")

block.zero_grad()
block.eval()
print(f"\nGradient flow: "
      f"{'PASSED — all params receive gradients' if not no_grad else 'FAILED: ' + str(no_grad)}")


## 3. SSM Internals Visualization

In [ ]:
# ── Test 6: A-matrix initial decay rates ─────────────────────────────────
# A[d,n] is initialised to -(n+1) so state n=0 has the longest memory
# and n=N-1 the shortest.  Retention = exp(A) in (0,1).

A_mat  = -torch.exp(block.mamba_block.A_log.float()).detach().cpu()  # (d_model, d_state)
retain = torch.exp(A_mat)   # values in (0,1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im = axes[0].imshow(retain.numpy(), aspect="auto", cmap="viridis", vmin=0, vmax=1)
axes[0].set_xlabel("State dim n")
axes[0].set_ylabel("Model dim d")
axes[0].set_title("State retention exp(A)\n[1 = perfect memory, 0 = instant forget]")
plt.colorbar(im, ax=axes[0])

axes[1].hist(retain.numpy().flatten(), bins=50, color="steelblue",
             edgecolor="white", linewidth=0.4)
axes[1].axvline(retain.mean().item(), color="red", linestyle="--",
                label=f"mean = {retain.mean():.3f}")
axes[1].set_xlabel("Retention rate")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Retention Rates")
axes[1].legend()

mean_per_n = retain.mean(dim=0).numpy()
axes[2].bar(range(config.d_state), mean_per_n,
            color=plt.cm.viridis(mean_per_n))
axes[2].set_xlabel("State dim n")
axes[2].set_ylabel("Mean retention")
axes[2].set_title("Mean Retention per State Dim\n(n=0: long memory → n=N-1: short)")
axes[2].set_xticks(range(config.d_state))

plt.suptitle("A-Matrix Decay Rates (at initialisation)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"{'n':>3}  {'Retention':>10}  {'Timescale (steps)':>18}  Bar")
print("-" * 50)
for n, r in enumerate(mean_per_n):
    ts  = -1.0 / (np.log(float(r)) + 1e-12)
    bar = "|" * int(r * 40)
    print(f"{n:>3}  {r:>10.4f}  {ts:>18.1f}  {bar}")


In [ ]:
# ── Test 7: Δ distribution and hidden-state dynamics ─────────────────────
# We manually replicate the SSM forward pass to capture intermediate tensors.

torch.manual_seed(42)
x_in = torch.randn(1, 64, config.d_input, device=device)
mb   = block.mamba_block

with torch.no_grad():
    # ── replicate forward up to SSM ──
    x  = mb.input_proj(x_in)
    L  = x.shape[1]
    xc = rearrange(x, "b l d -> b d l")
    xc = mb.conv1d(xc)[:, :, :L]
    xc = F.silu(rearrange(xc, "b d l -> b l d"))

    # ── SSM internals ──
    A     = -torch.exp(mb.A_log.float())
    delta = F.softplus(mb.dt_proj(mb.x_dt_proj(xc)))   # (1, L, d_model)
    B_p   = mb.x_B_proj(xc)                             # (1, L, d_state)

    dA  = torch.exp(torch.einsum("bld,dn->bldn", delta, A))
    dBu = torch.einsum("bld,bln->bldn", delta, B_p) * xc.unsqueeze(-1)

    gates  = rearrange(dA,  "b l d n -> b (d n) l").contiguous()
    tokens = rearrange(dBu, "b l d n -> b (d n) l").contiguous()
    h = warp_scan(gates, tokens)
    h = rearrange(h, "b (d n) l -> b l d n",
                  d=config.d_model, n=config.d_state)

delta_np = delta[0].cpu().numpy()     # (L, d_model)
h_np     = h[0].cpu().numpy()         # (L, d_model, d_state)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

im1 = axes[0, 0].imshow(delta_np.T, aspect="auto", cmap="plasma")
axes[0, 0].set_xlabel("Sequence position t")
axes[0, 0].set_ylabel("Model dim d")
axes[0, 0].set_title("Δ(t, d) — larger = more input-driven state update")
plt.colorbar(im1, ax=axes[0, 0])

axes[0, 1].hist(delta_np.flatten(), bins=60, color="darkorange",
                edgecolor="white", linewidth=0.3)
axes[0, 1].set_xlabel("Δ value")
axes[0, 1].set_ylabel("Count")
axes[0, 1].set_title(f"Δ distribution   "
                     f"mean={delta_np.mean():.3f}  std={delta_np.std():.3f}")

im2 = axes[1, 0].imshow(h_np[:, 0, :].T, aspect="auto", cmap="RdBu_r")
axes[1, 0].set_xlabel("Sequence position t")
axes[1, 0].set_ylabel("State dim n")
axes[1, 0].set_title("Hidden state h[t, d=0, :] — how memory evolves")
plt.colorbar(im2, ax=axes[1, 0])

h_norm = np.linalg.norm(h_np.reshape(L, -1), axis=-1)
axes[1, 1].plot(h_norm, color="teal", linewidth=1.5)
axes[1, 1].fill_between(range(L), 0, h_norm, alpha=0.2, color="teal")
axes[1, 1].set_xlabel("Sequence position t")
axes[1, 1].set_ylabel("||h[t]|| (all dims)")
axes[1, 1].set_title("Hidden state norm over time")

plt.suptitle("SSM Internal Dynamics (random input)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 4. Learning Task: AR(1) Next-Step Prediction

Train a minimal model — **embed → ResidualBlock → project** — to predict the
next value in a first-order autoregressive process:

> **x[t] = 0.9 · x[t-1] + ε[t]**,   ε ~ N(0, 0.01)

A memoryless baseline that always predicts zero achieves `MSE ≈ Var(x)`.
The SSM should significantly beat this by maintaining a running state estimate.


In [ ]:
# ── AR(1) model ───────────────────────────────────────────────────────────
@dataclass
class SmallConfig:
    d_input:     int  = 32
    d_model:     int  = 64
    d_state:     int  = 16
    dt_rank:     int  = 4
    kernel_size: int  = 4
    bias:        bool = False
    conv_bias:   bool = True

class AR1Predictor(nn.Module):
    def __init__(self, cfg):  # embed scalar -> ResidualBlock -> scalar
        super().__init__()
        self.embed   = nn.Linear(1, cfg.d_input)
        self.block   = ResidualBlock(cfg)
        self.project = nn.Linear(cfg.d_input, 1)

    def forward(self, x):
        # x: (B, L, 1)
        return self.project(self.block(self.embed(x)))

# ── Data generator ────────────────────────────────────────────────────────
def generate_ar1(batch, length, coef=0.9, noise=0.1, device="cpu"):
    eps = torch.randn(batch, length, 1, device=device) * noise
    x   = torch.zeros(batch, length, 1, device=device)
    x[:, 0] = eps[:, 0]
    for t in range(1, length):
        x[:, t] = coef * x[:, t - 1] + eps[:, t]
    return x

# ── Preview ───────────────────────────────────────────────────────────────
samples = generate_ar1(4, 128, device=device)
print(f"Data shape: {samples.shape}  mean={samples.mean():.4f}  std={samples.std():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

for i in range(4):
    axes[0].plot(samples[i, :, 0].cpu().numpy(), alpha=0.7, linewidth=1)
axes[0].set_title("Sample AR(1) sequences  (coef=0.9)")
axes[0].set_xlabel("t")
axes[0].set_ylabel("x[t]")

s    = samples[0, :, 0].cpu().numpy()
lags = range(25)
acf  = [1.0] + [float(np.corrcoef(s[:-k], s[k:])[0, 1]) for k in range(1, 25)]
axes[1].bar(lags, acf, color="steelblue", width=0.7)
axes[1].plot(lags, [0.9**k for k in lags], "r--", linewidth=1.5,
             label="Theoretical 0.9^k")
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_title("Autocorrelation Function")
axes[1].set_xlabel("Lag k")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── Training ──────────────────────────────────────────────────────────────
torch.manual_seed(0)
cfg   = SmallConfig()
model = AR1Predictor(cfg).to(device)
optim = torch.optim.Adam(model.parameters(), lr=1e-3)

SEQ_LEN = 64    # must be power-of-2 for accelerated_scan.warp
BATCH   = 32
STEPS   = 600

train_losses = []
val_log      = []   # (step, val_loss)

# Naive baseline: always predict 0  →  loss = Var(x)
with torch.no_grad():
    bs = generate_ar1(1000, SEQ_LEN + 1, device=device)
    baseline = F.mse_loss(torch.zeros_like(bs[:, 1:]), bs[:, 1:]).item()
print(f"Naive baseline MSE (predict zero): {baseline:.5f}")
print()

model.train()
for step in range(STEPS):
    seq   = generate_ar1(BATCH, SEQ_LEN + 1, device=device)
    x_in  = seq[:, :-1]   # (B, L, 1)  — inputs
    x_tgt = seq[:, 1:]    # (B, L, 1)  — next-step targets

    pred = model(x_in)
    loss = F.mse_loss(pred, x_tgt)

    optim.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optim.step()
    train_losses.append(loss.item())

    if (step + 1) % 100 == 0:
        model.eval()
        with torch.no_grad():
            v  = generate_ar1(128, SEQ_LEN + 1, device=device)
            vl = F.mse_loss(model(v[:, :-1]), v[:, 1:]).item()
        val_log.append((step + 1, vl))
        model.train()
        print(f"Step {step + 1:4d}  train={loss.item():.5f}  val={vl:.5f}"
              f"  improvement={baseline / vl:.2f}x over naive")

print("\nTraining complete.")


In [ ]:
# ── Results ───────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.3)

# (a) Loss curve
ax_l = fig.add_subplot(gs[0, :])
ax_l.plot(train_losses, color="steelblue", linewidth=0.6, alpha=0.45,
          label="Train (per step)")
W = 30
smooth = np.convolve(train_losses, np.ones(W) / W, mode="valid")
ax_l.plot(range(W - 1, STEPS), smooth, color="navy", linewidth=2,
          label=f"Smoothed (w={W})")
if val_log:
    vx, vy = zip(*val_log)
    ax_l.scatter(vx, vy, s=60, color="red", zorder=5, label="Val")
ax_l.axhline(baseline, color="gray", linestyle=":", linewidth=1.5,
             label=f"Naive baseline ({baseline:.4f})")
ax_l.set_xlabel("Training step")
ax_l.set_ylabel("MSE Loss")
ax_l.set_title("AR(1) Next-Step Prediction — Training Curve")
ax_l.set_yscale("log")
ax_l.legend()

# (b) & (c) Prediction vs ground truth on held-out sequences
model.eval()
with torch.no_grad():
    tseq = generate_ar1(4, SEQ_LEN + 1, device=device)
    pred = model(tseq[:, :-1]).cpu().numpy()          # (4, L, 1)
    true = tseq[:, 1:, 0].cpu().numpy()               # (4, L)

for idx, pos in enumerate([(1, 0), (1, 1)]):
    ax = fig.add_subplot(gs[pos[0], pos[1]])
    t  = range(SEQ_LEN)
    ax.plot(t, true[idx],          color="steelblue",  linewidth=1.5,
            label="Ground truth")
    ax.plot(t, pred[idx, :, 0],    color="darkorange", linewidth=1.5,
            linestyle="--", label="Predicted")
    mse = float(np.mean((true[idx] - pred[idx, :, 0]) ** 2))
    ax.set_title(f"Test sequence {idx + 1}   MSE={mse:.5f}")
    ax.set_xlabel("t")
    ax.legend(fontsize=8)

plt.suptitle("AR(1) Prediction Results", fontsize=14, y=1.01)
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────
final_val = val_log[-1][1] if val_log else float("nan")
print(f"{'=' * 48}")
print(f"{'Final validation MSE:':<30} {final_val:.5f}")
print(f"{'Naive baseline MSE:':<30} {baseline:.5f}")
print(f"{'Improvement over naive:':<30} {baseline / final_val:.2f}x")
print(f"{'Model parameters:':<30} {sum(p.numel() for p in model.parameters()):,}")
print(f"{'=' * 48}")
